## Fine-mapping comparisons

In [ ]:
git clone https://github.com/zhwm/RSparsePro_LD.git
cd RSparsePro_LD
pip install -r requirements.txt 

## DENTIST and SLALOM

In [ ]:
[finemap_DEN_SLA]
parameter: genofile = paths
# pheno_file: give genotype file (in plink)，we can read the gentype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
# specify the number of causal variants
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}_DEN_SLALOM.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container 
    ############################################################
    # 0. Load data
    ############################################################
    
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)
        # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                             full.names = TRUE, pattern = "\\.R$")) source(f)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    path = ${_input:r}
    dat <- readRDS(path)
    
    
    n        <- dat$n_samples
    n_snps   <- dat$n_snp
    ture_vars = dat$true_vars

    compute_LD_from_sketch <- function(U, p) {
      U <- as.matrix(U)
    
      # Orient so that rows = B pseudo-individuals, columns = p variants
      if (ncol(U) == p) {
        Uv <- U            # already B x p  (U = W^T X form)
      } else if (nrow(U) == p) {
        Uv <- t(U)         # stored p x B -> transpose to B x p
      } else {
        stop(sprintf(
          "Sketch U (%d x %d) matches neither dimension to variant count p = %d",
          nrow(U), ncol(U), p))
      }
    
      R <- cor(Uv)                 # p x p, diagonal exactly 1
      R[is.na(R)] <- 0             # guard: zero-variance variant columns
      R
    }

    
    ############################################################
    # 1. Helper: RAISS imputation on outliers
    ############################################################
    
    run_raiss_on_outliers <- function(outlier_flags, z_vec, ref_df, LD_mat) {
    
      known_idx <- which(!outlier_flags)
    
    
      known_df <- data.frame(
        chrom      = ref_df$chrom[known_idx],
        pos        = ref_df$pos[known_idx],
        variant_id = ref_df$variant_id[known_idx],
        A1         = ref_df$A1[known_idx],
        A2         = ref_df$A2[known_idx],
        z          = z_vec[known_idx],
        stringsAsFactors = FALSE
      )
    
      result <- raiss(
        ref_panel     = ref_df,
        known_zscores = known_df,
        LD_matrix     = LD_mat,
        lamb = 0.01,
        rcond = 0.01,
        R2_threshold = 0.6,
        minimum_ld   = 5,
        verbose      = FALSE
      )
    
      combined_z <- z_vec
    
      if (!is.null(result) && !is.null(result$result_nofilter)) {
        raiss_df <- result$result_nofilter
    
        for (i in which(outlier_flags)) {
          vid <- ref_df$variant_id[i]
          ri  <- which(raiss_df$variant_id == vid)
    
          if (length(ri) == 1 && !is.na(raiss_df$z[ri])) {
            combined_z[i] <- raiss_df$z[ri]
          }
        }
      }
    
      list(z = combined_z, raiss_result = result)
    }
    
    
    
    ############################################################
    # 2. Run QC methods (DENTIST + SLALOM)
    ############################################################
    
    
    finemap_QC <- function(input, n, name) {
          z  <- input$sumstat$z
          p  <- length(z)
    
          # LD reference: use provided LD, else derive it from the genotype sketch U
          if (!is.null(input$LD)) {
            LD <- input$LD
          } else if (!is.null(input$U)) {
            cat(sprintf("No LD in input; deriving R_hat = cor(U) from sketch (p = %d)\n", p))
            LD <- compute_LD_from_sketch(input$U, p)
          } else {
            stop("input must contain either 'LD' or 'U' (genotype sketch)")
          }
          ref_df   <- input$sumstat
    
    
                # DENTIST
        dentist_sw <- dentistSingleWindow(
          zScore  = z,
          R       = LD,
          nSample = n
        )
            dentist_outlier_flags <- dentist_sw$outlier
    
            # SLALOM
            slalom_result   <- slalom(zScore = z, R = LD)
            slalom_outliers <- slalom_result$data$outliers
            slalom_outliers[is.na(slalom_outliers)] <- FALSE
    
    
            ############################################################
            # 3. Run RAISS imputations
            ############################################################
    
            cat(sprintf("SLALOM+RAISS: imputing %d variants\n", sum(slalom_outliers)))
            slalom_raiss  <- run_raiss_on_outliers(slalom_outliers, z, ref_df, LD)
    
            cat(sprintf("DENTIST+RAISS: imputing %d variants\n", sum(dentist_outlier_flags)))
            dentist_raiss <- run_raiss_on_outliers(dentist_outlier_flags, z, ref_df, LD)
    
    
            ############################################################
            # 4. Construct QC-adjusted z-scores
            ############################################################
    
            # 4.1 DENTIST only
            dentist_qc_z <- z
            dentist_qc_z[dentist_outlier_flags] <- dentist_sw$imputed_z[dentist_outlier_flags]
    
            # 4.2 SLALOM + RAISS
            slalom_raiss_z <- slalom_raiss$z
    
            # 4.3 DENTIST + RAISS
            dentist_raiss_z <- dentist_raiss$z
    
    
            ############################################################
            # 5. Summary statistics
            ############################################################
    
            cat("=== Three-Way Comparison ===\n")
            cat(sprintf("Total variants: %d\n\n", n_snps))
    
            cat("Outlier detection:\n")
            cat(sprintf("  DENTIST: %d (%.1f%%)\n",
                        sum(dentist_outlier_flags), 100 * mean(dentist_outlier_flags)))
    
            cat(sprintf("  SLALOM:  %d (%.1f%%)\n",
                        sum(slalom_outliers), 100 * mean(slalom_outliers)))
    
            cat(sprintf("  Overlap: %d\n\n",
                        sum(dentist_outlier_flags & slalom_outliers)))
    
    
    
    
            ############################################################
            # 7. SuSiE fine-mapping comparison
            ############################################################
    
            library(susieR)
    
            run_susie <- function(z_vec, LD, n) {
              fit <- susie_rss(
                z           = z_vec,
                R           = LD,
                n           = n,
                control = susie_rss_control(
                    check_prior = FALSE
                )
              )
              fit
            }
    
    
            dentist_qc_res = run_susie(dentist_qc_z, LD, n)
            dentist_qc_res$outlier_number = sum(dentist_outlier_flags)
    
            dentist_raiss_res = run_susie(dentist_raiss_z, LD, n)
            dentist_raiss_res$outlier_number = sum(dentist_outlier_flags)
    
            slalom_raiss = run_susie(slalom_raiss_z, LD, n)
            print(sum(slalom_outliers) )
            slalom_raiss$outlier_number =  sum(slalom_outliers)
    
            output = list(dentist_qc_res = dentist_qc_res, dentist_raiss_res = dentist_raiss_res, slalom_raiss = slalom_raiss, input_path = path)
    
             return(output)
    }
    
    
    
        result_list <- list()
    
        result_list[["gtex"]] <- finemap_QC(dat$insample_input, n = n, name = "gtex")
        result_list[["kg"]]   <- finemap_QC(dat$kg_input,   n = n, name = "kg")
        result_list[["ADSP"]] <- finemap_QC(dat$ADSP_input, n = n, name = "ADSP")
        result_list[["UKB"]]  <- finemap_QC(dat$UKB_input,  n = n, name = "UKB")
    
        output = list(result_list = result_list, n_sample = n, input_path = path,
                  true_vars = dat$true_vars, n_snp = dat$n_snp)
        saveRDS(output, ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="Gtex_fine_DEN_SLALOM"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/11.Fig3A.Other_Methods.ipynb" finemap_DEN_SLA \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_simulation/result/h2g_H2G/causal_CAUSAL/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 1 2 3; do
  for h2g in 0.02 0.03 0.05 0.1 0.15; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

        sb "${output_script}"   # use sb if you aliased
  done
done

In [ ]:
[DEN_SLA_summary]
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}_METHOD.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)

    ## ---- helpers -------------------------------------------------------------
    # number of TRUE causal variants captured across all credible sets
    discover_var_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      length(intersect(variant, unlist(cs)))
    }
    # number of credible sets that contain >= 1 true causal variant
    true_set_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      sum(vapply(cs, function(s) length(intersect(variant, s)) > 0, logical(1)))
    }
    # pull a numeric field out of a file path, NA if absent (handles negatives, e.g. miss_fac_-1.0)
    parse_field <- function(f, key) {
      as.numeric(str_match(f, paste0(key, "_(-?[0-9.]+)"))[, 2])
    }

    ## ---- config --------------------------------------------------------------
    path  <- ${folder:r}
    files <- list.files(path, pattern = "sample_.*\\.rds$",
                        recursive = TRUE, full.names = TRUE)

    ## ---- detect which method scheme is present -------------------------------
    known_schemes <- list(
      flip      = c("gtex_original", "gtex_flip"),
      reference = c("ADSP", "gtex", "kg", "UKB")
    )

    stopifnot(length(files) > 0)
    available <- names(readRDS(files[[1]])[["result_list"]])

    scheme <- Filter(function(s) all(s %in% available), known_schemes)
    susie_methods <- if (length(scheme) >= 1) {
      scheme[[1]]                       # first fully-matching known scheme
    } else {
      available                         # fallback: whatever is actually in the file
    }
    message("Using susie_methods: ", paste(susie_methods, collapse = ", "))

    # QC pipelines: label -> where its credible set / outlier count live in the fit
    qc_specs <- list(
      DENTIST_QC    = list(cs    = function(s) s$dentist_qc_res$sets$cs,
                           n_out = function(s) s$dentist_qc_res$outlier_number),
      DENTIST_RAISS = list(cs    = function(s) s$dentist_raiss_res$sets$cs,
                           n_out = function(s) s$dentist_raiss_res$outlier_number),
      SLALOM_RAISS  = list(cs    = function(s) s$slalom_raiss$sets$cs,
                           n_out = function(s) s$slalom_raiss$outlier_number)
    )

    ## ---- build one long table, reading each file only once -------------------
    summary_tb <- map_dfr(files, function(f) {
      res     <- readRDS(f)
      variant <- res$true_vars
      meta <- tibble(
        input_file = f,
        ncausal    = parse_field(f, "ncausal"),
        miss_prop  = parse_field(f, "miss_prop"),
        miss_fac   = parse_field(f, "miss_fac")
      )
      map_dfr(susie_methods, function(m) {
        sus <- res[["result_list"]][[m]]
        if (is.null(sus)) return(NULL)
        map_dfr(names(qc_specs), function(qc) {
          cs    <- qc_specs[[qc]]$cs(sus)
          n_out <- tryCatch(qc_specs[[qc]]$n_out(sus), error = function(e) NULL)
          bind_cols(meta, tibble(
            method           = m,
            qc               = qc,
            outlier_number   = n_out %||% NA_real_,
            true_var_number  = discover_var_num(variant, cs),
            true_set_number  = true_set_num(variant, cs),
            total_var_number = length(variant),
            total_set_number = length(cs),
            total_set_length = length(unlist(cs))
          ))
        })
      })
    })

    ## ---- write one file per QC pipeline (same names as before) ---------------
    out_path <- ${_output:r}
    walk(names(qc_specs), function(qc_label) {
      saveRDS(filter(summary_tb, qc == qc_label),
              str_replace(out_path, "METHOD", qc_label))
    })

    # also keep the combined long table (handy for facet plotting original vs flip)
    saveRDS(summary_tb, str_replace(out_path, "METHOD", "ALL"))

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine_DEN_SLALOM/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/11.Fig3A.Other_Methods.ipynb" DEN_SLA_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

## CARMA 

In [ ]:
[finemap_CARMA]
parameter: genofile = paths
# pheno_file: give genotype file (in plink)，we can read the gentype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
# specify the number of causal variants
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container 
    
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)
    library(CARMA)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    

    compute_LD_from_sketch <- function(U, p) {
      U <- as.matrix(U)
    
      # Orient so that rows = B pseudo-individuals, columns = p variants
      if (ncol(U) == p) {
        Uv <- U            # already B x p  (U = W^T X form)
      } else if (nrow(U) == p) {
        Uv <- t(U)         # stored p x B -> transpose to B x p
      } else {
        stop(sprintf(
          "Sketch U (%d x %d) matches neither dimension to variant count p = %d",
          nrow(U), ncol(U), p))
      }
    
      R <- cor(Uv)                 # p x p, diagonal exactly 1
      R[is.na(R)] <- 0             # guard: zero-variance variant columns
      R
    }
    
    path = ${_input:r}
    
    dat = readRDS(path)
    
    n <- dat$n_samples
    
    finemap_CARMA <- function(input, n, name) {
      z  <- input$sumstat$z
      p  <- length(z)  
      # LD reference: use provided LD, else derive it from the genotype sketch U
      if (!is.null(input$LD)) {
        LD <- input$LD
      } else if (!is.null(input$U)) {
        cat(sprintf("No LD in input; deriving R_hat = cor(U) from sketch (p = %d)\n", p))
        LD <- compute_LD_from_sketch(input$U, p)
      } else {
        stop("input must contain either 'LD' or 'U' (genotype sketch)")
      }
        
     z.list      <- list(z)
     ld.list     <- list(as.matrix(LD))
     lambda.list <- list(1)
    
     CARMA.results <- CARMA(
      z.list,
      ld.list,
      lambda.list = lambda.list,
      outlier.switch = TRUE
     )
        CARMA.results$sets = list()

        if(length(CARMA.results[[1]]$`Credible set`[[2]])!=0){
            real_set <- CARMA.results[[1]]$`Credible set`[[2]]
        }else{
            real_set = NULL
        }
        
        CARMA.results$sets$cs = real_set
      
        return(CARMA.results)
    }
    
    result_list <- list()
    
    result_list[["gtex"]] <- finemap_CARMA(dat$insample_input, n = n, name = "gtex")
    result_list[["kg"]]   <- finemap_CARMA(dat$kg_input,   n = n, name = "kg")
    result_list[["ADSP"]] <- finemap_CARMA(dat$ADSP_input, n = n, name = "ADSP")
    result_list[["UKB"]]  <- finemap_CARMA(dat$UKB_input,  n = n, name = "UKB")
    output = list(result_list = result_list, n_sample = n, input_path = path, 
              true_vars = dat$true_vars, n_snp = dat$n_snp)
    saveRDS(output, ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="Gtex_fine_CARMA"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/11.Fig3A.Other_Methods.ipynb" finemap_CARMA \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_simulation/result/h2g_H2G/causal_CAUSAL/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 1 2 3; do
  for h2g in 0.02 0.03 0.05 0.1 0.15; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

        sb "${output_script}"   # use sb if you aliased
  done
done

In [ ]:
[CARMA_summary]
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}_summary.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)

    true_set_num <- function(variant, cs) {
      if (length(cs) == 0) {
        return(0L)
      }

      true_set <- integer(length(cs))

      for (i in seq_along(cs)) {
        true_set[i] <- as.integer(length(intersect(variant, cs[[i]])) > 0)
      }

      sum(true_set)
    }

    discover_var_num <- function(variant, cs) {
      if (length(cs) == 0) {
        return(0L)
      }
      length(intersect(variant, unlist(cs)))
    }

    path  <- ${folder:r}
    files <- list.files(path, pattern = "sample_.*\\.rds$", recursive = TRUE, full.names = TRUE)

    susie_methods <- c(
      "ADSP",
      "gtex",
      "kg",
      "UKB"
    )

    tb <- map_dfr(files, function(f) {
      res <- readRDS(f)
      variant <- res$true_vars
      ncausal <- as.numeric(sub(".*ncausal_([0-9.]+).*", "\\1", f))

      map_dfr(susie_methods, function(m) {
        sus <- res[["result_list"]][[m]]
        if(!is.null(sus$fit$set)){
            sus$fit$sets = sus$fit$set
        }
        cs  <- sus$fit$sets$cs

        tibble(
          input_file       = f,
          method           = m,
          ncausal          = ncausal,
          true_var_number  = discover_var_num(variant, cs),
          true_set_number  = true_set_num(variant, cs),
          total_var_number = length(variant),
          total_set_number = length(cs),
          total_set_length = length(unlist(cs))
        )
      })
    })

    saveRDS(tb, ${_output:r})

In [ ]:
work_dir="/home/sunh4/rss_qc/2026_output/Fine_CARMA/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash
#SBATCH --time=16:00
#SBATCH --mem=30G
#SBATCH --job-name=susie_summary
#SBATCH --cpus-per-task=2
#SBATCH --output=WORK_DIR/summary/summary.%j.out
#SBATCH --error=WORK_DIR/summary/summary.%j.err

export PATH=$HOME/miniforge3/envs/space/bin:$PATH

sos run /home/sunh4/rss_qc/2026_Simulation/4.Other_Methods.ipynb CARMA_summary \
    --folder WORK_DIR/result/ \
    --mem 20G \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sbatch ${output_script}

## RSparsePro

In [ ]:
[finemap_rsparse_pro]
parameter: genofile = paths
# pheno_file: give genotype file (in plink)，we can read the gentype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
# specify the number of causal variants
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}.rsparsepro.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container 

    library(tidyverse)

    path = ${_input:r}

    dat  <- readRDS(path)

    compute_LD_from_sketch <- function(U, p) {
      U <- as.matrix(U)
    
      # Orient so that rows = B pseudo-individuals, columns = p variants
      if (ncol(U) == p) {
        Uv <- U            # already B x p  (U = W^T X form)
      } else if (nrow(U) == p) {
        Uv <- t(U)         # stored p x B -> transpose to B x p
      } else {
        stop(sprintf(
          "Sketch U (%d x %d) matches neither dimension to variant count p = %d",
          nrow(U), ncol(U), p))
      }
    
      R <- cor(Uv)                 # p x p, diagonal exactly 1
      R[is.na(R)] <- 0             # guard: zero-variance variant columns
      R
    }
    
    finemap_Rsparse <- function(input,
                                out_prefix,
                                true_vars = NULL,
                                input_file = NULL,
                                python_bin = "python",
                                rsparsepro_script = "/sc/arion/projects/ad-omics/haochen/rss_qc/RSparsePro_LD/src/rsparsepro_ld.py") {
      z  <- input$sumstat$z
      p  <- length(z)  
      # LD reference: use provided LD, else derive it from the genotype sketch U
      if (!is.null(input$LD)) {
        LD <- input$LD
      } else if (!is.null(input$U)) {
        cat(sprintf("No LD in input; deriving R_hat = cor(U) from sketch (p = %d)\n", p))
        LD <- compute_LD_from_sketch(input$U, p)
      } else {
        stop("input must contain either 'LD' or 'U' (genotype sketch)")
      }
        
      LD <- as.matrix(LD)
      rsid <- colnames(LD)
      if (is.null(rsid)) {
        rsid <- paste0("snp_", seq_along(z))
      }
    
      z_df <- tibble(
        RSID = rsid,
        Z = z
      )
    
      out_prefix = ${_output:r}
    
      z_file      <- paste0(out_prefix, ".z.txt")
      ld_file     <- paste0(out_prefix, ".ld.txt")
      result_path <- paste0(out_prefix, ".rsparsepro.txt")
      rds_path    <- paste0(out_prefix, ".rsparsepro.rds")
    
      write_tsv(z_df, z_file)
    
      write.table(
        LD,
        file = ld_file,
        sep = " ",
        row.names = FALSE,
        col.names = FALSE,
        quote = FALSE
      )
    
      cmd <- paste(
        shQuote(python_bin),
        shQuote(rsparsepro_script),
        "--z", shQuote(z_file),
        "--ld", shQuote(ld_file),
        "--save", shQuote(out_prefix)
      )
    
      status <- system(cmd)
    
      res <- read_tsv(result_path, show_col_types = FALSE) %>%
        mutate(index = row_number())
    
      if (!"cs" %in% names(res)) {
        stop("RSparsePro output does not contain column 'cs'.")
      }
    
      cs_index <- setdiff(unique(res$cs), 0)
    
      result_rds <- list()
      result_rds$fit <- res
      result_rds$set <- list()
    
      if (length(cs_index) == 0) {
        result_rds$set$cs <- NULL
      } else {
        cs <- vector("list", length(cs_index))
        for (i in seq_along(cs_index)) {
          cs[[i]] <- res %>%
            filter(cs == cs_index[i]) %>%
            pull(index)
        }
        result_rds$set$cs <- cs
      }
    
      return(result = list(fit = result_rds))
    }
    
    
    result_gtex <- finemap_Rsparse(
      input      = dat$insample_input,
      out_prefix = "gtex",
      true_vars  = dat$true_vars,
      input_file = path
    )
    
    result_kg <- finemap_Rsparse(
      input      = dat$kg_input,
      out_prefix = "kg",
      true_vars  = dat$true_vars,
      input_file = path
    )
    
    result_ADSP <- finemap_Rsparse(
      input      = dat$ADSP_input,
      out_prefix = "ADSP",
      true_vars  = dat$true_vars,
      input_file = path
    )
    
    result_UKB <- finemap_Rsparse(
      input      = dat$UKB_input,
      out_prefix = "UKB",
      true_vars  = dat$true_vars,
      input_file = path
    )

    result_list <- list()
    
    result_list[["gtex"]] <- result_gtex
    result_list[["kg"]]   <- result_kg
    result_list[["ADSP"]] <- result_ADSP
    result_list[["UKB"]]  <- result_UKB

    output = list(result_list = result_list, n_sample = dat$n_samples, input_path = path, 
              true_vars = dat$true_vars, n_snp = dat$n_snp)
    saveRDS(output, ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="Gtex_fine_Rsparse_Pro"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/11.Fig3A.Other_Methods.ipynb" finemap_rsparse_pro \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_simulation/result/h2g_H2G/causal_CAUSAL/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 1 2 3; do
  for h2g in 0.02 0.03 0.05 0.1 0.15; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

        sb "${output_script}"   # use sb if you aliased
  done
done

## Summary

In [ ]:
[single_QC_summary]
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)

    ## ---- helpers -------------------------------------------------------------
    discover_var_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      length(intersect(variant, unlist(cs)))
    }
    true_set_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      sum(vapply(cs, function(s) length(intersect(variant, s)) > 0, logical(1)))
    }
    parse_field <- function(f, key) {
      as.numeric(str_match(f, paste0(key, "_(-?[0-9.]+)"))[, 2])
    }
    # single-QC credible set: CARMA stores it at $sets$cs, RSparsePro at $fit$set$cs
    get_cs <- function(sus) {
      if (is.null(sus))             return(NULL)
      if (!is.null(sus$sets$cs))    return(sus$sets$cs)      # CARMA
      if (!is.null(sus$set$cs))     return(sus$set$cs)       # (plain set$cs)
      if (!is.null(sus$fit$set$cs)) return(sus$fit$set$cs)   # RSparsePro
      NULL
    }

    ## ---- config --------------------------------------------------------------
    path  <- ${folder:r}
    files <- list.files(path, pattern = "sample_.*\\.rds$",
                        recursive = TRUE, full.names = TRUE)
    stopifnot(length(files) > 0)

    # detect which method scheme is present in the results
    known_schemes <- list(
      flip      = c("gtex_original", "gtex_flip"),
      reference = c("ADSP", "gtex", "kg", "UKB")
    )
    available <- names(readRDS(files[[1]])[["result_list"]])
    scheme    <- Filter(function(s) all(s %in% available), known_schemes)
    susie_methods <- if (length(scheme) >= 1) scheme[[1]] else available
    message("Using susie_methods: ", paste(susie_methods, collapse = ", "))

    ## ---- build summary (each file read once) ---------------------------------
    summary_tb <- map_dfr(files, function(f) {
      res     <- readRDS(f)
      variant <- res$true_vars
      meta <- tibble(
        input_file = f,
        ncausal    = parse_field(f, "ncausal"),
        miss_prop  = parse_field(f, "miss_prop"),
        miss_fac   = parse_field(f, "miss_fac")
      )
      map_dfr(susie_methods, function(m) {
        sus <- res[["result_list"]][[m]]
        if (is.null(sus)) return(NULL)
        cs <- get_cs(sus)
        bind_cols(meta, tibble(
          method           = m,
          true_var_number  = discover_var_num(variant, cs),
          true_set_number  = true_set_num(variant, cs),
          total_var_number = length(variant),
          total_set_number = length(cs),
          total_set_length = length(unlist(cs))
        ))
      })
    })

    saveRDS(summary_tb, ${_output:r})

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine_CARMA/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/11.Fig3A.Other_Methods.ipynb" single_QC_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine_Rsparse_Pro/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/11.Fig3A.Other_Methods.ipynb" single_QC_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}